# MK & Adipocyte evaluation: based on the segmented objects
- use the centroids of the segmented objects to match with the ground truth objects
- calculate the confusion matrix based on the matched objects
- visualize the confusion matrix as a heatmap

In [ ]:
import numpy as np
import pandas as pd

import os
from datetime import datetime

from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_bipartite_matching

In [ ]:
marker = 'adipocyte'  # 'MK' or 'adipocyte'
age = 'd30' # d30, 'd60', '2yo', '3mo', '1yo'

# Under the subfolder we get the files end with _positions.csv, and read them and concat them together
df_cell_positions = pd.DataFrame()
if marker == 'MK':
    positions_folder = 'data_prediction_metric/MKs'
elif marker == 'adipocyte':
    positions_folder = 'data_prediction_metric/Adipocytes'
for fname in os.listdir(positions_folder):
    # print("Checking:", fname)
    if fname.endswith("_Position.csv") and age in fname:
        print("Reading:", fname)
        df_part = pd.read_csv(os.path.join(positions_folder, fname), skiprows=3)
        print(f"  -> {len(df_part)} rows")
        if 'iMK' in fname:
            df_part['cell_type'] = 'iMK'
        elif 'pMK' in fname:
            df_part['cell_type'] = 'pMK'
        elif 'iAdipo'in fname:
            df_part['cell_type'] = 'iAdipo'
        elif 'pAdipo'in fname:
            df_part['cell_type'] = 'pAdipo'
        else:
            print(f"[WARN] Could not determine cell type from filename: {fname}")
            df_part['cell_type'] = 'unknown'
        df_cell_positions = pd.concat([df_cell_positions, df_part], ignore_index=True)


In [ ]:
# Fix the origin shift for different ages and markers to match the images
if marker == 'MK':
    if age == '2yo':
        df_cell_positions['Position X'] -= 67427.8
        df_cell_positions['Position Y'] -= 40664.2
        df_cell_positions['Position Z'] -= 0
    elif age == 'd30':
        df_cell_positions['Position X'] -= 59842.1
        df_cell_positions['Position Y'] -= 40680.9
        df_cell_positions['Position Z'] -= -183.068
    elif age == 'd60':
        df_cell_positions['Position X'] -= 58391.5
        df_cell_positions['Position Y'] -= 42608.5
        df_cell_positions['Position Z'] -= -109.25
elif marker == 'adipocyte':
    # no shift for age 3mo and 1yo
    if age == '2yo':
        df_cell_positions['Position X'] -= 67427.8
        df_cell_positions['Position Y'] -= 40664.2
        df_cell_positions['Position Z'] -= 0
    elif age == 'd30':
        df_cell_positions['Position X'] -= 68203.5
        df_cell_positions['Position Y'] -= 43696
        df_cell_positions['Position Z'] -= 0
    elif age == 'd60':
        df_cell_positions['Position X'] -= 54384.1
        df_cell_positions['Position Y'] -= 41343.7
        df_cell_positions['Position Z'] -= -169.762
# Now we build up a KD-tree for the MK positions to enable fast nearest neighbor search
if marker == 'MK':
    df_cell_positions_p = df_cell_positions[df_cell_positions['cell_type'] == 'pMK'].copy()
    df_cell_positions_i = df_cell_positions[df_cell_positions['cell_type'] == 'iMK'].copy()
    print(f"Total MKs: {len(df_cell_positions)}, pMKs: {len(df_cell_positions_p)}, iMKs: {len(df_cell_positions_i)}")

elif marker == 'adipocyte':
    df_cell_positions_p = df_cell_positions[df_cell_positions['cell_type'] == 'pAdipo'].copy()
    df_cell_positions_i = df_cell_positions[df_cell_positions['cell_type'] == 'iAdipo'].copy()
    print(f"Total Adipocytes: {len(df_cell_positions)}, pAdipocytes: {len(df_cell_positions_p)}, iAdipocytes: {len(df_cell_positions_i)}")


In [ ]:
# --------------------------------------------------
# Settings
# --------------------------------------------------
distance_threshold = 10.0 # 10 µm threshold
df_cell_positions_i = df_cell_positions_i.reset_index(drop=True).copy()
df_cell_positions_p = df_cell_positions_p.reset_index(drop=True).copy()

# Coordinates
coords_i = df_cell_positions_i[['Position X', 'Position Y', 'Position Z']].to_numpy()
coords_p = df_cell_positions_p[['Position X', 'Position Y', 'Position Z']].to_numpy()

n_i = len(coords_i)
n_p = len(coords_p)

# --------------------------------------------------
# Step 1: KD-trees
# --------------------------------------------------
tree_i = cKDTree(coords_i)
tree_p = cKDTree(coords_p)

# --------------------------------------------------
# Step 2: Find all candidate pairs within threshold
# candidate_pairs[i] = list of predicted indices within threshold of imaged i
# --------------------------------------------------
candidate_pairs = tree_i.query_ball_tree(tree_p, r=distance_threshold)

rows = []
cols = []
dists = []

for idx_i, pred_list in enumerate(candidate_pairs):
    for idx_p in pred_list:
        dist = np.linalg.norm(coords_i[idx_i] - coords_p[idx_p])
        rows.append(idx_i)
        cols.append(idx_p)
        dists.append(dist)

rows = np.array(rows, dtype=int)
cols = np.array(cols, dtype=int)
dists = np.array(dists, dtype=float)

# Safety check
if len(rows) == 0:
    print("No candidate matches found within threshold.")
    
    df_tp = pd.DataFrame(columns=['idx_i', 'idx_p', 'distance_um'])

    fn_idx_i = np.arange(n_i)
    fp_idx_p = np.arange(n_p)

    df_fn = df_cell_positions_i.copy()
    df_fn['match_label'] = 'FN'
    df_fn['idx_i'] = fn_idx_i

    df_fp = df_cell_positions_p.copy()
    df_fp['match_label'] = 'FP'
    df_fp['idx_p'] = fp_idx_p

else:
    # --------------------------------------------------
    # Step 3: Build adjacency matrix
    # rows = imaged MK, cols = predicted MK
    # --------------------------------------------------
    adj = csr_matrix(
        (np.ones(len(rows), dtype=int), (rows, cols)),
        shape=(n_i, n_p)
    )

    # --------------------------------------------------
    # Step 4: Maximum 1-to-1 bipartite matching
    # IMPORTANT: perm_type='column' returns array indexed by rows
    # match_for_i[i] = matched predicted index, or -1
    # --------------------------------------------------
    match_for_i = maximum_bipartite_matching(adj, perm_type='column')

    # matched row indices
    matched_i = np.where(match_for_i != -1)[0]
    matched_p = match_for_i[matched_i].astype(int)

    # --------------------------------------------------
    # Step 5: Recover distances robustly
    # --------------------------------------------------
    pair_to_dist = {(int(i), int(p)): float(d) for i, p, d in zip(rows, cols, dists)}

    # This should now work
    matched_distances = np.array(
        [pair_to_dist[(int(i), int(p))] for i, p in zip(matched_i, matched_p)],
        dtype=float
    )

    # TP table
    df_tp = pd.DataFrame({
        'Position X': df_cell_positions_i.loc[matched_i, 'Position X'],
        'Position Y': df_cell_positions_i.loc[matched_i, 'Position Y'],
        'Position Z': df_cell_positions_i.loc[matched_i, 'Position Z'],
        'ID': df_cell_positions_i.loc[matched_i, 'ID'],
        'idx_i': matched_i,
        'idx_p': matched_p,
        'distance_um': matched_distances,
        'match_label': 'TP',
        
    })

    # FN = imaged only
    fn_idx_i = np.where(match_for_i == -1)[0]
    df_fn = df_cell_positions_i.iloc[fn_idx_i].copy()
    df_fn['match_label'] = 'FN'
    df_fn['idx_i'] = fn_idx_i
    df_fn['Position X'] = df_cell_positions_i.loc[fn_idx_i, 'Position X']
    df_fn['Position Y'] = df_cell_positions_i.loc[fn_idx_i, 'Position Y']
    df_fn['Position Z'] = df_cell_positions_i.loc[fn_idx_i, 'Position Z']
    df_fn['ID'] = df_cell_positions_i.loc[fn_idx_i, 'ID']

    # FP = predicted only
    matched_p_set = set(matched_p.tolist())
    fp_idx_p = np.array([j for j in range(n_p) if j not in matched_p_set], dtype=int)

    df_fp = df_cell_positions_p.iloc[fp_idx_p].copy()
    df_fp['match_label'] = 'FP'
    df_fp['idx_p'] = fp_idx_p
    df_fp['Position X'] = df_cell_positions_p.loc[fp_idx_p, 'Position X']
    df_fp['Position Y'] = df_cell_positions_p.loc[fp_idx_p, 'Position Y']
    df_fp['Position Z'] = df_cell_positions_p.loc[fp_idx_p, 'Position Z']
    df_fp['ID'] = df_cell_positions_p.loc[fp_idx_p, 'ID']

# --------------------------------------------------
# Optional: label original dataframes
# --------------------------------------------------
df_cell_positions_i_labeled = df_cell_positions_i.copy()
df_cell_positions_i_labeled['match_label'] = 'FN'
df_cell_positions_i_labeled['matched_pred_idx'] = -1
df_cell_positions_i_labeled['match_distance_um'] = np.nan

if len(df_tp) > 0:
    df_cell_positions_i_labeled.loc[df_tp['idx_i'], 'match_label'] = 'TP'
    df_cell_positions_i_labeled.loc[df_tp['idx_i'], 'matched_pred_idx'] = df_tp['idx_p'].values
    df_cell_positions_i_labeled.loc[df_tp['idx_i'], 'match_distance_um'] = df_tp['distance_um'].values

df_cell_positions_p_labeled = df_cell_positions_p.copy()
df_cell_positions_p_labeled['match_label'] = 'FP'
df_cell_positions_p_labeled['matched_imaged_idx'] = -1
df_cell_positions_p_labeled['match_distance_um'] = np.nan

if len(df_tp) > 0:
    df_cell_positions_p_labeled.loc[df_tp['idx_p'], 'match_label'] = 'TP'
    df_cell_positions_p_labeled.loc[df_tp['idx_p'], 'matched_imaged_idx'] = df_tp['idx_i'].values
    df_cell_positions_p_labeled.loc[df_tp['idx_p'], 'match_distance_um'] = df_tp['distance_um'].values

# --------------------------------------------------
# Summary
# --------------------------------------------------
print(f"Total imaged {marker}s     : {n_i}")
print(f"Total predicted {marker}s  : {n_p}")
print(f"Matched TP pairs     : {len(df_tp)}")
print(f"False negatives (FN) : {len(df_fn)}")
print(f"False positives (FP) : {len(df_fp)}")

In [ ]:
# caculate the precision, recall, and F1-score based on the counts of TP, FP, and FN
precision = len(df_tp) / (len(df_tp) + len(df_fp)) if (
    len(df_tp) + len(df_fp)) > 0 else 0.0
recall = len(df_tp) / (len(df_tp) + len(df_fn)) if (
    len(df_tp) + len(df_fn)) > 0 else 0.0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1_score:.4f}")

In [ ]:
# Save a csv containing the TP, FP, and FN, precision, recall, and F1-score and also total counts
summary_df = pd.DataFrame({
    'match_label': ['TP', 'FP', 'FN', 'precision', 'recall', 'f1_score'],
    'count': [len(df_tp), len(df_fp), len(df_fn), precision, recall, f1_score],
})
results_dir = "results_bone_age"
time_marker = datetime.now().strftime("%y%m%d")
out_summary_csv = os.path.join(results_dir, f"{time_marker}_YL_{marker}_matching_summary_dist{distance_threshold}_{age}.csv")
summary_df.to_csv(out_summary_csv, index=False)
print(f"Saved {marker} matching summary CSV:", out_summary_csv)